# Constat Demo: HR Raise Analysis

This notebook demonstrates a complete HR raise analysis workflow using Constat's
Jupyter magics interface. It connects to a running Constat server, analyzes
employee performance data, generates raise recommendations, and creates
personalized email communications.

## Prerequisites

1. Start the server: `constat serve -c demo/config.yaml`
2. Install the Jupyter extension: `pip install -e constat-jupyter`
3. If auth is enabled, run `%constat login` below before connecting

In [ ]:
%load_ext constat_jupyter

In [ ]:
%constat login

In [ ]:
%constat connect sales-analytics,hr-reporting

## Step 1: Generate Raise Recommendations

Ask Constat to analyze performance reviews and generate raise recommendations
using the business rules documented in the HR domain.

In [ ]:
%%constat

Estimate employee raises for each current employee.
Save the results to a table called raise_recommendations with columns:
employee_name, department, job_title, current_salary, rating,
raise_pct, raise_amount, new_salary.

1. Look up each employee's most recent performance review
2. Use that review's manager rating to select the corresponding
   min/max raise percentage range from the performance review
   guidelines table in business_rules.md
3. Analyze the sentiment of the manager's review comments, assign a score
   from 0 to 1, and interpolate a raise percentage between that min and max

Ignore the compensation policy.

In [ ]:
%constat tables

In [ ]:
%constat table raise_recommendations

## Internal Analysis

Raise distribution and policy compliance side by side.

In [ ]:
%%constat auto

Compare the recommended raises in raise_recommendations against the
compensation policy in business_rules.md. For each employee, show their
rating, the policy min/max raise range for that rating, and the
recommended raise percentage. Flag any employees whose recommended raise
falls outside their policy range.

Save results to a table called policy_comparison with columns:
employee_name, department, rating, policy_min_pct, policy_max_pct,
raise_pct, in_range (boolean).

In [ ]:
%%constat auto

Compare current employee salaries and proposed post-raise salaries
against typical startup/VC-funded company salary ranges for equivalent
roles and experience levels. Show which employees are above, at, or
below market rates.

Save results to a table called market_comparison with columns:
employee_name, job_title, current_salary, new_salary,
market_median, market_position (above/at/below).

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Load both tables ---
pc = _constat_session.table("policy_comparison", pandas=True)
mc = _constat_session.table("market_comparison", pandas=True)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Raises vs Policy Bands", "Current vs Post-Raise vs Market"),
    horizontal_spacing=0.08,
)

# --- Left: Policy comparison scatter ---
pc_name = next((c for c in pc.columns if "name" in c.lower()), pc.columns[0])
pc_rating = next((c for c in pc.columns if "rating" in c.lower()), None)
pc_raise = next((c for c in pc.columns if "raise" in c.lower() and "pct" in c.lower()), None)
pc_min = next((c for c in pc.columns if "min" in c.lower()), None)
pc_max = next((c for c in pc.columns if "max" in c.lower()), None)
pc_flag = next((c for c in pc.columns if "range" in c.lower() or "flag" in c.lower()), None)

if all([pc_rating, pc_raise, pc_min, pc_max]):
    for rating in sorted(pc[pc_rating].unique()):
        sub = pc[pc[pc_rating] == rating]
        lo, hi = sub[pc_min].iloc[0], sub[pc_max].iloc[0]
        fig.add_hrect(
            y0=lo, y1=hi, fillcolor="green", opacity=0.08,
            annotation_text=f"Rating {rating}: {lo}-{hi}%",
            annotation_position="top left",
            row=1, col=1,
        )
    colors = ["green" if v else "red" for v in pc[pc_flag]] if pc_flag else "steelblue"
    fig.add_trace(go.Scatter(
        x=pc[pc_name], y=pc[pc_raise],
        mode="markers", marker=dict(size=10, color=colors),
        text=pc.apply(lambda r: f"Rating: {r[pc_rating]}, Raise: {r[pc_raise]:.1f}%", axis=1),
        hoverinfo="text+x", showlegend=False,
    ), row=1, col=1)
    fig.update_xaxes(tickangle=-45, row=1, col=1)
    fig.update_yaxes(title_text="Raise %", row=1, col=1)

# --- Right: Market comparison grouped bars ---
mc_name = next((c for c in mc.columns if "name" in c.lower()), mc.columns[0])
mc_curr = next((c for c in mc.columns if "current" in c.lower() and "sal" in c.lower()), None)
mc_new = next((c for c in mc.columns if "new" in c.lower() and "sal" in c.lower()), None)
mc_market = next((c for c in mc.columns if "market" in c.lower() and "median" in c.lower()), None)

if all([mc_curr, mc_new, mc_market]):
    fig.add_trace(go.Bar(name="Current", x=mc[mc_name], y=mc[mc_curr]), row=1, col=2)
    fig.add_trace(go.Bar(name="Post-Raise", x=mc[mc_name], y=mc[mc_new]), row=1, col=2)
    fig.add_trace(go.Bar(name="Market Median", x=mc[mc_name], y=mc[mc_market]), row=1, col=2)
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    fig.update_yaxes(title_text="Salary ($)", row=1, col=2)

fig.update_layout(height=500, width=1200, barmode="group")
fig.show()

## Raise Distribution

In [ ]:
import plotly.express as px

df = _constat_session.table("raise_recommendations", pandas=True)

pct_col = next((c for c in df.columns if "raise" in c.lower() and "pct" in c.lower()), None)
dept_col = next((c for c in df.columns if "dept" in c.lower() or "department" in c.lower()), None)

if pct_col and dept_col:
    fig = px.histogram(
        df, x=pct_col, color=dept_col, nbins=15,
        title="Raise % Distribution by Department",
        labels={pct_col: "Raise %", dept_col: "Department"},
        barmode="overlay", opacity=0.7,
    )
    fig.show()

    avg = df.groupby(dept_col)[pct_col].mean().reset_index()
    fig2 = px.bar(
        avg, x=dept_col, y=pct_col,
        title="Average Raise % by Department",
        labels={pct_col: "Avg Raise %", dept_col: "Department"},
        text_auto=".1f",
    )
    fig2.show()
else:
    print(f"Expected columns not found. Available: {list(df.columns)}")

## Executive Summary

Generate a CFO-ready summary with budget impact and department breakdown.

In [ ]:
%%constat auto include:md

Create an executive summary of the raise recommendations
for CFO approval. Include total budget impact, department
breakdown, and distribution by raise percentage tier.

## Employee Communications

Generate personalized raise announcement emails. The `published` flag
displays only starred (key result) tables.

In [ ]:
%%constat auto published

Generate a personalized email communication for each employee
announcing their raise. Save to a table called employee_emails
with columns: employee_name, email_address, html_email_body

In [ ]:
# Preview email table and sample
df = _constat_session.table("employee_emails")
print(f"employee_emails: {len(df)} rows")
display(df.head(5))

row = df.to_pandas().iloc[0]
html_col = next((c for c in row.index if "html" in c.lower() and "body" in c.lower()), None)
if html_col:
    from IPython.display import HTML
    display(HTML(f"<h4>Sample email for {row.iloc[0]}:</h4>"))
    display(HTML(row[html_col]))

## Export for Mail Merge

In [ ]:
df = _constat_session.table("employee_emails", pandas=True)
df.to_csv("raise_emails.csv", index=False)
print(f"Exported {len(df)} emails to raise_emails.csv")